# Machine learning labs · Check your setup
Run this short notebook in local Jupyter, MIN JupyterHub, Google Colab, or the
course Nix environment. It is self-contained: **no lab folder or helper file is
needed**. Python, NumPy, and Matplotlib are required. PyTorch and a GPU are optional.

In a Git checkout, copy this notebook from `setup` into the ignored `my-work`
folder before running it. Create that folder at the course root if necessary.
In Colab, upload this notebook alone. No companion files are needed.

Run cells from top to bottom with **Shift+Enter** and check that a plot appears.
For a clean check, restart the kernel/runtime and run all cells in order.
Save your notebook copy when you finish. In Colab, save or download it so that
you keep your work after the runtime ends. Nothing is installed or downloaded.


## 1. Python and required libraries
The minimum versions are Python **3.10**, NumPy **1.24** and Matplotlib **3.7**.
If a required check fails, use the setup instructions for your chosen environment,
then restart the kernel/runtime and rerun this notebook.


In [ ]:
import sys
import re

if sys.version_info[:2] < (3, 10):
    raise RuntimeError(
        f"Python 3.10 or newer is required; this kernel uses {sys.version.split()[0]}. "
        "Select the Python environment described in the setup guide and restart the kernel."
    )

try:
    import numpy as np
    import matplotlib
    import matplotlib.pyplot as plt
except ImportError as error:
    raise ImportError(
        "A required library could not be imported. Lab 1 needs NumPy and Matplotlib. "
        "Follow the setup guide for your environment, then restart the kernel/runtime. "
        f"Original import error: {error}"
    ) from error


def release_numbers(version):
    """Read leading major/minor/patch integers, ignoring build or release suffixes."""
    match = re.match(r"^\s*(\d+)\.(\d+)(?:\.(\d+))?", str(version))
    if match is None:
        raise RuntimeError(f"Cannot interpret library version {version!r}; ask for setup help.")
    return tuple(int(part) if part is not None else 0 for part in match.groups())


for name, version, minimum in (
    ("NumPy", np.__version__, (1, 24, 0)),
    ("Matplotlib", matplotlib.__version__, (3, 7, 0)),
):
    if release_numbers(version) < minimum:
        required = ".".join(map(str, minimum))
        raise RuntimeError(
            f"{name} {required} or newer is required; this kernel uses {version}. "
            "Follow the setup guide and restart the kernel/runtime after changing the environment."
        )

print("Python:", sys.version.split()[0])
print("Python executable:", sys.executable)
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)
print("Required versions: OK")
environment_ok = True


## 2. A small calculation and a plot
You should see a matrix with shape `(3, 2)` and a plot of a parabola.
The arrays are deliberately small and run on the CPU.


In [ ]:
matrix_a = np.arange(1.0, 7.0).reshape(3, 2)
matrix_b = 2.0 * np.eye(2)
matrix_product = matrix_a @ matrix_b
assert matrix_product.shape == (3, 2), "Unexpected matrix-product shape."
assert np.isfinite(matrix_product).all(), "The calculation produced non-finite values."
np.testing.assert_allclose(matrix_product, [[2, 4], [6, 8], [10, 12]])
print("Matrix product:\n", matrix_product)
print("Shape:", matrix_product.shape)

plot_x = np.linspace(-1.0, 1.0, 101)
figure, axis = plt.subplots(figsize=(5, 3))
axis.plot(plot_x, plot_x**2)
axis.set(xlabel="x", ylabel="x²", title="Matplotlib setup check")
figure.tight_layout()
plt.show()
print("Calculation and plotting calls: OK. Check that the plot is visible above.")
numpy_plot_ok = True


In [ ]:
assert environment_ok and numpy_plot_ok
print("SETUP SUCCESS: all required setup checks passed.")
print("If the plot is visible, your environment is ready for the first lab.")
print("You do not need PyTorch or a GPU for Lab 1.")


## 3. Optional: PyTorch and CUDA
This section is informative. Missing PyTorch, unavailable CUDA or an optional
backend error does **not** invalidate the successful Lab 1 checks above.
If CUDA is available, a tiny matrix multiplication checks that computation really
runs on the GPU. Nothing is installed or downloaded.


In [ ]:
try:
    import torch
except ImportError as error:
    print("Optional PyTorch: not available in this environment.")
    print("This is acceptable: Lab 1 uses NumPy on the CPU.")
    print("Import information:", error)
except Exception as error:
    print(f"Optional PyTorch could not load: {type(error).__name__}: {error}")
    print("The required Lab 1 checks above are unaffected.")
else:
    print("Optional PyTorch version:", torch.__version__)
    print("CUDA version reported by PyTorch:", torch.version.cuda)
    gpu_inputs = gpu_result = None
    try:
        cuda_available = torch.cuda.is_available()
        print("CUDA available:", cuda_available)
        if cuda_available:
            device = torch.device("cuda:0")
            with torch.no_grad():
                gpu_inputs = torch.arange(16, dtype=torch.float32, device=device).reshape(4, 4)
                gpu_result = gpu_inputs @ gpu_inputs.T
                torch.cuda.synchronize(device)
                if gpu_result.device.type != "cuda" or tuple(gpu_result.shape) != (4, 4):
                    raise RuntimeError("The optional calculation did not return the expected CUDA tensor.")
                if not torch.isfinite(gpu_result).all().item():
                    raise RuntimeError("The optional GPU calculation produced non-finite values.")
                expected = np.arange(16, dtype=np.float32).reshape(4, 4)
                np.testing.assert_allclose(gpu_result.cpu().numpy(), expected @ expected.T)
            print("GPU:", torch.cuda.get_device_name(device))
            print("Small GPU matrix multiplication: OK on", gpu_result.device)
        else:
            print("No available CUDA GPU. CPU computation is sufficient for Lab 1.")
    except Exception as error:
        print(f"Optional CUDA check did not succeed: {type(error).__name__}: {error}")
        print("The required Lab 1 checks above are unaffected; continue with NumPy on the CPU.")
    finally:
        # Remove references to the tiny GPU tensors, including after a failed check.
        del gpu_result, gpu_inputs


If a required check failed, save or copy the error message and the reported
versions when asking for help. If only the optional section reported a problem,
you can still complete Lab 1. Save your notebook before closing the session.
